## Imports

In [25]:
import pytesseract
import cv2
import os
import re
import numpy as np
import pandas as pd

if "notebooks" in os.getcwd():
    os.chdir("../../../")

## Helper Functions

### Label Cleaning

In [26]:
def clean_label_text(raw_text):
    """
    Standardizes OCR output to match this format: <TYPE [Number]-[Number]>
    """
    # Map common OCR misreadings to our species label types
    type_map = {
        "PHYCA": "PHYCA", "PHY": "PHYCA", "PHICA": "PHYCA", "BYCA": "PHYCA",
        "VAU": "VAU", "VAV": "VAU", "VAVU": "VAU", "VAY": "VAU",
        "PEH": "PEH", "PEM": "PEH", "REH": "PEH", "PED": "PEH",
        "CAT": "CAT"
    }

    raw_text = raw_text.upper()

    # Extract the species
    matched_type = "UNKNOWN"
    for key, val in type_map.items():
        if key in raw_text:
            matched_type = val
            break

    # Find the number pattern using regex
    pattern = r'(\d+[\s\.\-\_]*\d+\s*[A-Z]?)'
    match = re.search(pattern, raw_text)

    if match:
        # Standardize the output to <TYPE Number-Number>
        nums = match.group(1).replace(" ", "").replace(".", "-").replace("_", "-").replace("--", "-")
        # Ensure only one hyphen remains
        nums = re.sub(r'-+', '-', nums)
        return f"{matched_type} {nums}"

    return matched_type if matched_type != "UNKNOWN" else raw_text

### Label Extraction

In [27]:
def extract_with_tesseract(image_path):
    """
    Extracts seed species label from the top, bottom, or sides of the image.
    """
    img = cv2.imread(image_path)
    if img is None: return ["Error"]
    h, w, _ = img.shape

    # Using 20% / 5% ROI strategy to isolate the label
    zones = {
        "top": img[0:int(h*0.12), :],
        "bottom": img[int(h*0.88):h, :],
        "left": img[:, 0:int(w*0.12)],
        "right": img[:, int(w*0.88):w]
    }

    final_results = []

    # Tesseract configuration:
    # --psm 6: assume a single uniform block of text
    # -c tessedit_char_whitelist: limit to capital letters and digits
    custom_config = r'--psm 11 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-_'

    # Rotate text depending on which part of the image it is found in
    for zone_name, crop in zones.items():
        if zone_name == "left":
            crop = cv2.rotate(crop, cv2.ROTATE_90_CLOCKWISE)
        elif zone_name == "right":
            crop = cv2.rotate(crop, cv2.ROTATE_90_COUNTERCLOCKWISE)
        elif zone_name == "bottom":
            crop = cv2.rotate(crop, cv2.ROTATE_180)

        # Preprocessing steps
        # 1. Grayscale
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

        # 2. Denoise using median blur to remove graininess in the cropped sections
        denoised = cv2.medianBlur(gray, 5)

        # 3. Simple thresholding (fixed threshold of 127 to force a clean black and white result)
        _, thresh = cv2.threshold(denoised, 127, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        # 4. OCR Scan
        text = pytesseract.image_to_string(thresh, config=custom_config)

        # DEBUG - temporary diagnostic show
        if "0044" in image_path:
            cv2.imwrite("diagnostic_thresh.jpg", thresh)

        if text.strip():
            cleaned = clean_label_text(text)
            if any(species in cleaned for species in ["PHYCA", "VAU", "PEH", "CAT"]):
                final_results.append(cleaned)

    return final_results

## Batch Image Processing

In [28]:
val_img_dir = "data/seed/images/val/"

val_images = [f for f in os.listdir(val_img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
all_labels = []

print(f"Processing {len(val_images)} images...")

for img_name in val_images:
    full_path = os.path.join(val_img_dir, img_name)
    labels = extract_with_tesseract(full_path)

    combined_label = " | ".join(labels)

    all_labels.append({
        "image_name": img_name,
        "extracted_text": combined_label
    })
    print(f"Done: {img_name} -> {combined_label}")

df_results = pd.DataFrame(all_labels)
print(df_results)

Processing 6 images...
Done: IMG_0044.jpg -> 
Done: IMG_0057.jpg -> 
Done: IMG_0059.jpg -> 
Done: IMG_0072.jpg -> PHYCA
Done: IMG_0082.jpg -> VAU 2-9

Done: IMG_0090.jpg -> 
     image_name extracted_text
0  IMG_0044.jpg               
1  IMG_0057.jpg               
2  IMG_0059.jpg               
3  IMG_0072.jpg          PHYCA
4  IMG_0082.jpg      VAU 2-9\n
5  IMG_0090.jpg               
